# 01 — Delphes output sanity checks

First look at the MPD pipeline output. The goal here is **"is the simulation
doing something sane"**, not physics. Concretely:

1. Does the file open and does it have the events we asked for?
2. Do the object multiplicities match the parton-level expectation?
   (`ttbar_semilep` → 4 jets, `ttbar_semilep_2j` → 6 jets, signal → 5 jets)
3. Do the b-tag multiplicities match? (both backgrounds → 2, signal → **1**)
4. Does the hadronic W show up at 80 GeV, and the lepton+MET transverse mass
   edge at 80 GeV? If those are wrong, something upstream is broken.

Sample paths are set in the second cell — point them at whatever
`run_mpd.sh` produced. Samples that are missing are just skipped, so this
notebook is usable with only the two backgrounds while the signal model is
still being sorted out.

In [ ]:
import os
import sys
from pathlib import Path

import awkward as ak
import matplotlib.pyplot as plt
import numpy as np

# Allow running from notebooks/ without installing the package.
sys.path.insert(0, str(Path.cwd().parent))

from bnv2 import delphes_io as dio
from bnv2 import kinematics as kin

plt.rcParams.update({
    "figure.figsize": (6.5, 4.2),
    "figure.dpi": 110,
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

WORK = Path(os.environ.get("BNV2_WORK", Path.cwd().parent / "work"))

# sample label -> Delphes ROOT file. Glob so you do not have to paste run tags.
SAMPLE_GLOBS = {
    "signal_bnv":       "signal_bnv/Events/*/tag_1_delphes_events.root",
    "ttbar_semilep":    "ttbar_semilep/Events/*/tag_1_delphes_events.root",
    "ttbar_semilep_2j": "ttbar_semilep_2j/Events/*/tag_1_delphes_events.root",
}

COLORS = {
    "signal_bnv": "#d62728",
    "ttbar_semilep": "#1f77b4",
    "ttbar_semilep_2j": "#2ca02c",
}

# Most recent matching file per sample.
SAMPLES = {}
for label, pattern in SAMPLE_GLOBS.items():
    matches = sorted(WORK.glob(pattern), key=lambda p: p.stat().st_mtime)
    if matches:
        SAMPLES[label] = matches[-1]
        print(f"{label:20s} {matches[-1]}")
    else:
        print(f"{label:20s} -- not found under {WORK / pattern}")

assert SAMPLES, f"no Delphes files found under {WORK}"

## Inventory

Before plotting anything, look at what is actually in the file. If the event
count is not what you asked for, or a collection you expect is missing from
the tree, stop here and fix the Delphes card.

In [ ]:
# What does the tree actually contain? (first sample only, they share a card)
first = next(iter(SAMPLES.values()))
tree = dio.open_tree(str(first))
print(f"{first.name}: {tree.num_entries} events\n")

prefixes = sorted({k.split(".")[0] for k in tree.keys() if "." in k})
print("collections in the tree:")
print("  " + ", ".join(prefixes))

print("\nbranches this notebook reads:")
for name, (prefix, members) in dio.DEFAULT_COLLECTIONS.items():
    have = all(f"{prefix}.{m}" in tree.keys() for m in members)
    print(f"  {name:12s} {prefix:12s} {'ok' if have else 'MISSING'}")

In [ ]:
raw = {}
for label, path in SAMPLES.items():
    raw[label] = dio.load_delphes(str(path))
    print(f"{label:20s} {len(raw[label]['jet']):6d} events")

## Object selection and derived quantities

One cell of helpers, so the plotting cells below stay short. Thresholds here
are loose CMS-like placeholders, not an optimized selection — the point is to
see the distributions, not to cut on them.

In [ ]:
JET_PT_MIN, JET_ETA_MAX = 30.0, 2.4
LEP_PT_MIN, LEP_ETA_MAX = 25.0, 2.4
M_W = 80.379


def derive(objs):
    """Apply loose object selection and compute the quantities we plot."""
    jets = objs["jet"]
    jets = jets[(jets.pt > JET_PT_MIN) & (abs(jets.eta) < JET_ETA_MAX)]
    jets = jets[ak.argsort(jets.pt, ascending=False)]

    leptons = ak.concatenate([objs["electron"], objs["muon"]], axis=1)
    leptons = leptons[(leptons.pt > LEP_PT_MIN) & (abs(leptons.eta) < LEP_ETA_MAX)]
    leptons = leptons[ak.argsort(leptons.pt, ascending=False)]

    btag = jets.btag == 1
    light = jets[~btag]

    d = {
        "jets": jets,
        "leptons": leptons,
        "n_jet": ak.num(jets, axis=1),
        "n_btag": ak.sum(btag, axis=1),
        "n_lep": ak.num(leptons, axis=1),
        "ht": kin.scalar_ht(jets.pt),
        "met": objs["met"].met,
        "met_phi": objs["met"].phi,
    }

    # Leading-object kinematics, None for events that have none.
    d["lead_jet_pt"] = ak.firsts(jets.pt)
    d["lead_jet_eta"] = ak.firsts(jets.eta)
    d["lead_lep_pt"] = ak.firsts(leptons.pt)

    # Hadronic W candidate: light-jet pair with mass closest to m_W.
    pairs = ak.combinations(light, 2, fields=["a", "b"])
    m_jj = kin.invariant_mass(pairs.a, pairs.b)
    best = ak.argmin(abs(m_jj - M_W), axis=1, keepdims=True)
    d["m_jj_best"] = ak.firsts(m_jj[best])

    # Transverse mass of leading lepton + MET. Should show the W Jacobian edge.
    d["mt_lep_met"] = kin.transverse_mass(
        d["lead_lep_pt"], ak.firsts(leptons.phi), d["met"], d["met_phi"]
    )

    # Mass of the whole visible jet system: a crude ttbar-scale check.
    d["m_alljets"] = kin.collection_mass(jets)
    return d


sel = {label: derive(objs) for label, objs in raw.items()}


def clean(x):
    """Drop None entries (events with no such object) and return numpy."""
    x = ak.flatten(x, axis=None) if getattr(x, "ndim", 1) > 1 else x
    return ak.to_numpy(x[~ak.is_none(x)])


def overlay(key, bins, xlabel, title=None, logy=False, ax=None):
    """Area-normalized overlay of one quantity across all loaded samples."""
    ax = ax or plt.gca()
    for label, d in sel.items():
        v = clean(d[key])
        if len(v) == 0:
            continue
        ax.hist(v, bins=bins, density=True, histtype="step", linewidth=1.8,
                color=COLORS.get(label), label=f"{label} ({len(v)})")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("a.u. (area normalized)")
    if title:
        ax.set_title(title)
    if logy:
        ax.set_yscale("log")
    ax.legend(fontsize=8)
    return ax

## Multiplicities

This is the plot that matters most for stage 1. Expected peak positions:

| | jets | b-tags |
|---|---|---|
| `signal_bnv` | 5 | **1** |
| `ttbar_semilep` | 4 | 2 |
| `ttbar_semilep_2j` | 6 | 2 |

Delphes b-tagging is a parameterized efficiency, so the b-tag distributions
will be smeared well below the truth values (~70% per b at a typical working
point means the 2-b samples peak at 1, not 2). Read the *relative* shift
between samples, not the absolute peak.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.0))

overlay("n_jet", np.arange(-0.5, 12.5), f"N jets (pT > {JET_PT_MIN:.0f} GeV, |eta| < {JET_ETA_MAX})",
        "Jet multiplicity", ax=axes[0])
overlay("n_btag", np.arange(-0.5, 6.5), "N b-tagged jets",
        "b-tag multiplicity", ax=axes[1])
overlay("n_lep", np.arange(-0.5, 4.5), f"N leptons (pT > {LEP_PT_MIN:.0f} GeV)",
        "Lepton multiplicity", ax=axes[2])

plt.tight_layout()
plt.show()

for label, d in sel.items():
    print(f"{label:20s} <N_jet> = {ak.mean(d['n_jet']):5.2f}   "
          f"<N_btag> = {ak.mean(d['n_btag']):5.2f}   "
          f"<N_lep> = {ak.mean(d['n_lep']):5.2f}")

## Basic kinematics

Nothing here should be surprising. Things to actually look at:

- **Leading jet eta** must be symmetric about 0 and must fall off at the
  tracker/calorimeter boundary set in the Delphes card. An asymmetry means a
  bug, not physics.
- **MET** should look essentially the same in all three samples. All of them
  have exactly one real neutrino. If the signal MET spectrum differs a lot,
  either the signal decay is not what you think it is, or something is
  mis-modeled.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

overlay("lead_jet_pt", np.linspace(0, 500, 50), "leading jet pT [GeV]",
        "Leading jet pT", ax=axes[0][0])
overlay("lead_jet_eta", np.linspace(-3, 3, 40), "leading jet eta",
        "Leading jet eta", ax=axes[0][1])
overlay("lead_lep_pt", np.linspace(0, 300, 40), "leading lepton pT [GeV]",
        "Leading lepton pT", ax=axes[0][2])
overlay("met", np.linspace(0, 400, 40), "MET [GeV]",
        "Missing ET", ax=axes[1][0])
overlay("ht", np.linspace(0, 1500, 50), "HT [GeV]",
        "Scalar jet pT sum", ax=axes[1][1])
overlay("m_alljets", np.linspace(0, 1500, 50), "m(all selected jets) [GeV]",
        "Invariant mass of the jet system", ax=axes[1][2])

plt.tight_layout()
plt.show()

## Resonance closure

These two are the real "is the pipeline working" test, because both have a
known answer.

- **m(jj) closest to 80 GeV** should show a clear peak at the W mass in all
  three samples — every one of them has a hadronic W. A peak that is shifted
  or very wide points at the jet energy scale/resolution in the Delphes card.
  (Note the "closest to m_W" choice biases toward 80 by construction, so
  judge the *width* and the tails, not the peak position alone.)
- **mT(lepton, MET)** should show the Jacobian edge at 80 GeV with a tail
  above it from resolution. If there is no edge, the lepton and the MET are
  not coming from the same W and you should check the decay chains.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

overlay("m_jj_best", np.linspace(0, 250, 50),
        "m(jj) closest to m$_W$ [GeV]", "Hadronic W candidate", ax=axes[0])
axes[0].axvline(M_W, color="k", ls="--", lw=1, alpha=0.6)

overlay("mt_lep_met", np.linspace(0, 200, 50),
        "m$_T$(lepton, MET) [GeV]", "Leptonic W transverse mass", ax=axes[1])
axes[1].axvline(M_W, color="k", ls="--", lw=1, alpha=0.6)

plt.tight_layout()
plt.show()

## Red flags checklist

Work through this before trusting anything downstream.

- [ ] Event count matches `nevents` in the run card.
- [ ] Jet multiplicity peaks where the parton counting says it should, one
      unit apart between `ttbar_semilep` and the signal, and two between the
      two backgrounds.
- [ ] `ttbar_semilep_2j` does **not** show a pile-up of jets right at the
      generator `ptj = 20` threshold that survives the 30 GeV offline cut —
      if it does, your generator cut is too close to your analysis cut.
- [ ] b-tag multiplicity is visibly lower in the signal than in both
      backgrounds. This is the whole discriminating handle; if it is not
      there, check that b is excluded from `j` in the proc cards.
- [ ] Leading jet eta is symmetric and cut off by the Delphes acceptance.
- [ ] MET spectra are compatible across all three samples.
- [ ] m(jj) peaks at 80 GeV and mT has an edge at 80 GeV.
- [ ] Lepton multiplicity is dominantly exactly 1.

If all of that holds, the pipeline is sound and the next step is a real
event selection plus the discriminating variables (b-tag count, the 4-jet
BNV top candidate mass, and the absence of a second b).